# Construct End-User Parquet Files

In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
from typing import List
import h5py
atl.ATLAS = "ColliderML"

sys.path.append("../")
# from utils import load_root_file, load_hepmc_event
from edm4hep_utils import build_calo_df, build_particle_df, build_tracker_df, load_edm4hep_file
from clustering_metrics import evaluate_clustering, plot_clustering_metrics

from edm4hep_utils import pixel_readouts, strip_readouts
all_tracker_readouts = pixel_readouts + strip_readouts

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Roadmap

1. Load in edm4hep file
2. Load in ambi tracks csv 
3. Load in particles root
4. Load in measurements root
5. Load in trackstates ambi root
6. Load in tracksummary ambi root
7. Create track parquet object
8. Track object needs:
    - hits in the track (hit ID)
    - track fit parameters
    - track quality parameters
    - track states at IP, end-of-track, first hit, last hit

# Loading

In [3]:
base_dir = "/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0"
event_num = 0

## 1. Load in edm4hep file

In [3]:
edm4hep_file = f"{base_dir}/edm4hep.root"
event = load_edm4hep_file(edm4hep_file, event_num=event_num)
events = uproot.open(edm4hep_file)["events"]

In [7]:
event.keys()

dict_keys(['tracker_df', 'calo_hits_df', 'calo_contrib_df', 'particles_df', 'parents_df', 'daughters_df'])

In [4]:
tracker_df = event["tracker_df"]
tracker_df.columns

Index(['cellID', 'EDep', 'time', 'pathLength', 'quality', 'x', 'y', 'z', 'px',
       'py', 'pz', 'particle_id', 'r', 'R', 'phi', 'theta', 'eta', 'pt',
       'detector'],
      dtype='object')

In [6]:
tracker_df.detector

0            PixelBarrelReadout
1            PixelBarrelReadout
2            PixelBarrelReadout
3            PixelBarrelReadout
4            PixelBarrelReadout
                  ...          
22060    LongStripEndcapReadout
22061    LongStripEndcapReadout
22062    LongStripEndcapReadout
22063    LongStripEndcapReadout
22064    LongStripEndcapReadout
Name: detector, Length: 22065, dtype: object

In [5]:
parents_df = event["parents_df"]
parents_df.columns

particles_df = event["particles_df"]
# Create a column from the index
particles_df["particle_id"] = particles_df.index
particles_df.columns

Index(['PDG', 'generatorStatus', 'simulatorStatus', 'charge', 'time', 'mass',
       'vx', 'vy', 'vz', 'px', 'py', 'pz', 'endpoint_x', 'endpoint_y',
       'endpoint_z', 'parents_begin', 'parents_end', 'daughters_begin',
       'daughters_end', 'pt', 'p', 'eta', 'phi', 'particle_id'],
      dtype='object')

# Hits Objects

In [12]:
def get_run_paths(base_dir: str) -> List[str]:
    """
    Get all run directories in the dataset, properly sorted.
    """
    # Get all run directories and sort numerically by run number
    run_dirs = glob.glob(f"{base_dir}/runs/*/")
    return sorted(run_dirs, key=lambda x: int(x.rstrip('/').split('/')[-1]))

def process_event_for_hits(
    event_id: int,
    tracker_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Process hit data for a single event using EDM4HEP tracker hits.
    
    Args:
        event_id: Event number
        tracker_df: DataFrame containing EDM4HEP tracker hits
        
    Returns:
        DataFrame containing hit data for this event
    """
    # Select relevant columns
    hit_columns = [
        "cellID",
        "EDep",
        "time",
        "x",
        "y",
        "z",
        "px",
        "py",
        "pz",
        "particle_id",
        "detector",
    ]
    
    event_hits = tracker_df[hit_columns].copy().rename(columns={"EDep": "energy"})
    
    # Add event_id
    event_hits["event_id"] = event_id
    
    return event_hits

def build_hdf5_hits(
    df: pd.DataFrame,
    output_file: str
) -> None:
    """
    Build HDF5 file with event/hit hierarchy.
    
    Structure:
    /events/
        /event_0/
            /hits    # Dataset containing hit properties
        /event_1/
            ...
    """
    with h5py.File(output_file, 'a') as f:
        # Create events group if it doesn't exist
        if 'events' not in f:
            events_group = f.create_group('events')
        else:
            events_group = f['events']
            
        # Group DataFrame by event_id
        for event_id, event_df in df.groupby('event_id'):
            # Create event group
            event_group = events_group.create_group(f'event_{event_id}')
            
            # Store hit data
            event_group.create_dataset(
                'hits',
                data=event_df.drop(columns=['event_id']).to_records(index=False),
                compression="gzip",
                compression_opts=9
            )

def process_full_dataset_for_hits(
    base_dir: str,
    output_base_dir: str,
    chunk_size: int = 1000,
    run_size: int = 10,
    dataset_name: str = "pileup-10/ttbar/v1/reco/hits/"
) -> None:
    """
    Process entire dataset in chunks.
    """
    # Get properly sorted run directories
    run_dirs = get_run_paths(base_dir)
    num_runs = len(run_dirs)
    num_events = num_runs * run_size
    
    # Calculate runs per chunk
    runs_per_chunk = chunk_size // run_size
    
    print(f"Processing {num_runs} runs with {num_events} total events")
    print(f"Processing {runs_per_chunk} runs per chunk to get ~{chunk_size} events per file")
    
    output_dir = f"{output_base_dir}/{dataset_name}"
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    dataset_name = dataset_name.replace("/", ".")
    
    # Process chunks of runs
    for start_run in tqdm(range(0, num_runs, runs_per_chunk), desc="Processing chunks"):
        all_events_data = []
        
        # Process each run in the chunk
        for run_idx in range(start_run, min(start_run + runs_per_chunk, len(run_dirs))):
            run_dir = run_dirs[run_idx]
            print(f"Processing run {run_idx} at {run_dir}")
            
            # Load EDM4HEP file
            edm4hep_file = f"{run_dir}/edm4hep.root"
            event = load_edm4hep_file(edm4hep_file, event_num=0)
            
            # Process event
            event_df = process_event_for_hits(
                run_idx * run_size,
                event["tracker_df"]
            )
            all_events_data.append(event_df)
            
        if all_events_data:
            combined_df = pd.concat(all_events_data, ignore_index=True)
            
            # Calculate event range for filename
            start_event = start_run * run_size
            end_event = min((start_run + runs_per_chunk) * run_size - 1, 
                           len(run_dirs) * run_size - 1)
            
            output_file = f"{output_dir}/{dataset_name}.events{start_event}-{end_event}.h5"
            build_hdf5_hits(combined_df, output_file)
            print(f"Saved {output_file}")

In [13]:
def read_event_hits(
    input_file: str,
    event_id: int
) -> pd.DataFrame:
    """
    Read hit data for a specific event.
    """
    with h5py.File(input_file, 'r') as f:
        event_group = f[f'events/event_{event_id}']
        
        # Read hit data
        hits = event_group['hits'][:]
        
        # Convert to DataFrame
        df = pd.DataFrame(hits)
        df['event_id'] = event_id
        return df

def read_events_hits(
    base_dir: str,
    event_ids: List[int],
    dataset_name: str = "mu10.ttbar"
) -> pd.DataFrame:
    """
    Read specific events from HDF5 files.
    """
    events_data = []
    chunk_size = 1000  # Must match the chunk size used in writing
    
    for event_id in event_ids:
        # Calculate which file contains this event
        chunk_num = event_id // chunk_size
        start_event = chunk_num * chunk_size
        
        # Find the file
        pattern = f"{base_dir}/{dataset_name}.events{start_event}-*.h5"
        matching_files = glob.glob(pattern)
        
        if not matching_files:
            print(f"Could not find file containing event {event_id}")
            continue
            
        filename = matching_files[0]
        
        try:
            event_df = read_event_hits(filename, event_id)
            events_data.append(event_df)
        except KeyError as e:
            print(f"Could not read event {event_id}: {e}")
            continue
    
    return pd.concat(events_data, ignore_index=True) if events_data else pd.DataFrame()

In [14]:
# Process the full dataset
base_dir = "/eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1"
output_base_dir = "/eos/home-d/dmurnane/ColliderML/staging/"
dataset_name = "pileup-10/ttbar/v1/truth/hits"


In [15]:
process_full_dataset_for_hits(base_dir, output_base_dir, dataset_name=dataset_name)

Processing 128 runs with 1280 total events
Processing 100 runs per chunk to get ~1000 events per file


Processing chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Processing run 0 at /eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/0/
Processing run 1 at /eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/1/
Processing run 2 at /eos/home-d/dmurnane/www/ColliderML/simulation/gg2ttbar/v1/runs/2/


Processing chunks:   0%|          | 0/2 [00:24<?, ?it/s]


KeyboardInterrupt: 

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../../scripts/postprocessing/")
from convert_hits import process_run_for_hits, process_chunk_for_hits, convert_hits
from utils.utils import get_run_paths, ensure_output_dir, get_chunk_info

In [6]:
run_number = 0
run_size = 10
hits = process_run_for_hits(
    base_dir,
    run_number,
    run_size
)


2025-02-04 03:28:50,797 - root - INFO - Processing run 0 from /global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0
2025-02-04 03:28:50,799 - root - INFO - Using EDM4HEP file: /global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0/edm4hep.root
2025-02-04 03:28:58,933 - root - INFO - Successfully processed 10/10 events in run 0


In [7]:
hits

[                cellID    energy        time           x            y  \
 0      277213933012998  0.000358    5.960443  -32.366956     1.492748   
 1      253850653166598  0.000463    5.775142    2.450572   -31.891473   
 2       16490694721078  0.001319   10.141488   41.731344  -165.855602   
 3       15941056347702  0.000005   10.134196   42.007489  -165.670502   
 4      263403717461766  0.000245    5.512754  -31.250231     6.906123   
 ...                ...       ...         ...         ...          ...   
 22060   16990891724846  0.000250   15.349312 -138.471589   879.620673   
 22061   16844861788478  0.000067   16.720472 -162.685917  1046.138671   
 22062   16922172248382  0.000073   16.738754 -163.004477  1048.359523   
 22063   16776142283102  0.000093  137.691208  745.627140   706.365467   
 22064     455267635278  0.000326   21.885483  -99.789754   872.090909   
 
                  z        px         py         pz  particle_id  \
 0      -401.924170 -0.331791   0.019908  

In [20]:
from pathlib import Path

input_base_dir = "/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/"
output_base_dir = "/global/cfs/cdirs/m4958/data/ColliderML/staging"
dataset_name = "pileup-10/ttbar/v1/truth/hits"
start_run = 100
runs_per_chunk = 100
run_size = 10
chunk_size = 1000


base_dir = Path(input_base_dir)
output_base_dir = Path(output_base_dir)

run_dirs = get_run_paths(base_dir)
num_runs = len(run_dirs)

num_events, runs_per_chunk, num_chunks = get_chunk_info(num_runs, run_size, chunk_size)
output_dir = ensure_output_dir(output_base_dir, dataset_name)
dataset_name = dataset_name.replace("/", ".")

process_chunk_for_hits(
    run_dirs,
    start_run,
    runs_per_chunk,
    Path(output_dir),
    dataset_name,
    run_size
    )

2025-02-05 05:15:32,590 - root - INFO - 
Processing chunk: events 1000-1279


end_run: 128


Processing runs:   0%|          | 0/28 [00:00<?, ?it/s]2025-02-05 05:15:32,592 - root - INFO - Processing run 100 from /global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/100
2025-02-05 05:15:32,592 - root - INFO - Using EDM4HEP file: /global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/100/edm4hep.root
2025-02-05 05:15:39,123 - root - INFO - Successfully processed 10/10 events in run 100
Processing runs:   4%|▎         | 1/28 [00:06<02:56,  6.53s/it]2025-02-05 05:15:39,124 - root - INFO - Processing run 101 from /global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/101
2025-02-05 05:15:39,125 - root - INFO - Using EDM4HEP file: /global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/101/edm4hep.root
2025-02-05 05:15:44,922 - root - INFO - Successfully processed 10/10 events in run 101
Processing run

In [21]:
run_dirs

[PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/0'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/1'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/2'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/3'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/4'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/5'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/6'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/Side_Work/ACTS/outputs/low_pileup_pilot/gg2ttbar/v1/runs/7'),
 PosixPath('/global/cfs/projectdirs/m3443/usr/dtmurnane/

In [17]:

# Read back specific events
events_df = read_events_hits(
    output_dir,
    event_ids=[1009, 1010],
    dataset_name=dataset_name
)

In [18]:
events_df

,cell_id,energy,time,x,y,z,px,py,pz,particle_id,detector,event_id
0,1720420207622,0.000129,-7.250226,6.964802,-31.347151,182.500437,0.389561,-1.795232,6.907298,4159,b'PixelBarrelReadout',1009
1,8381897706758,0.000173,-7.233591,7.235965,-32.597263,187.309032,0.389212,-1.795907,6.906769,4159,b'PixelBarrelReadout',1009
2,269655646869782,0.000143,-6.783647,14.305111,-66.548609,317.648183,0.364200,-1.800776,6.903570,4159,b'PixelBarrelReadout',1009
3,30029036464422,0.000161,-6.177980,23.301244,-112.391738,493.071643,0.340731,-1.805586,6.898878,4159,b'PixelBarrelReadout',1009
4,262233758764294,0.000322,-7.063366,12.739115,-29.499498,239.737026,0.737134,-1.683603,10.186787,4158,b'PixelBarrelReadout',1009
...,...,...,...,...,...,...,...,...,...,...,...,...
27379,386547155038,0.000158,10.553947,-804.332400,-13.232334,2995.500000,-0.035103,-0.074414,0.150519,258120,b'LongStripEndcapReadout',1010
27380,16045999009806,0.000377,13.949459,-49.521067,-784.896462,1279.500000,-0.015107,0.053273,0.018887,258266,b'LongStripEndcapReadout',1010
27381,16363826593822,0.000339,17.726517,-42.348266,-792.581781,1590.500000,-0.021396,0.047512,0.014652,258266,b'LongStripEndcapReadout',1010
27382,16252156395550,0.000265,17.789951,-47.694757,-775.116190,1595.500000,-0.005406,0.047521,0.014002,258266,b'LongStripEndcapReadout',1010
